In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Dense, GlobalMaxPooling1D, Embedding, Dropout, Attention, LayerNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import numpy as np

In [14]:


# Replace the placeholder with your data
data = """Mr. Phileas Fogg lived, in 1872, at No. 7, Saville Row, Burlington Gardens, the house in which Sheridan died in 1814.
He was one of the most noticeable members of the Reform Club, though he seemed always to avoid attracting attention; an enigmatical personage, about whom little was known, except that he was a polished man of the world.
People said that he resembled Byron--at least that his head was Byronic; but he was a bearded, tranquil Byron, who might live on a thousand years without growing old.
Certainly an Englishman, it was more doubtful whether Phileas Fogg was a Londoner.
He was never seen on 'Change, nor at the Bank, nor in the counting-rooms of the "City"; no ships ever came into London docks of which he was the owner; he had no public employment; he had never been entered at any of the Inns of Court, either at the Temple, or Lincoln's Inn, or Gray's Inn; nor had his voice ever resounded in the Court of Chancery, or in the Exchequer, or the Queen's Bench, or the Ecclesiastical Courts.
He certainly was not a manufacturer; nor was he a merchant or a gentleman farmer.
His name was strange to the scientific and learned societies, and he never was known to take part in the sage deliberations of the Royal Institution or the London Institution, the Artisan's Association, or the Institution of Arts and Sciences.
He belonged, in fact, to none of the numerous societies which swarm in the English capital, from the Harmonic to that of the Entomologists, founded mainly for the purpose of abolishing pernicious insects.
Was Phileas Fogg rich? Undoubtedly. But those who knew him best could not imagine how he had made his fortune, and Mr. Fogg was the last person to whom to apply for the information.He was not lavish, nor, on the contrary, avaricious; for, whenever he knew that money was needed for a noble, useful, or benevolent purpose, he supplied it quietly and sometimes anonymously. He was, in short, the least communicative of men.
He talked very little, and seemed all the more mysterious for his taciturn manner. His daily habits were quite open to observation; but whatever he did was so exactly the same thing that he had always done before, that the wits of the curious were fairly puzzled.
Had he travelled? It was likely, for no one seemed to know the world more familiarly; there was no spot so secluded that he did not appear to have an intimate acquaintance with it. He often corrected, with a few clear words, the thousand conjectures advanced by members of the club as to lost and unheard-of travellers, pointing out the true probabilities, and seeming as if gifted with a sort of second sight, so often did events justify his predictions. He must have travelled everywhere, at least in the spirit.
It was at least certain that Phileas Fogg had not absented himself from London for many years. Those who were honoured by a better acquaintance with him than the rest, declared that nobody could pretend to have ever seen him anywhere else. His sole pastimes were reading the papers and playing whist.
He often won at this game, which, as a silent one, harmonised with his nature; but his winnings never went into his purse, being reserved as a fund for his charities.
Mr. Fogg played, not to win, but for the sake of playing. The game was in his eyes a contest, a struggle with a difficulty, yet a motionless, unwearying struggle, congenial to his tastes.
Phileas Fogg was not known to have either wife or children, which may happen to the most honest people; either relatives or near friends, which is certainly more unusual. He lived alone in his house in Saville Row, whither none penetrated. A single domestic sufficed to serve him. He breakfasted and dined at the club, at hours mathematically fixed, in the same room, at the same table, never taking his meals with other members, much less bringing a guest with him; and went home at exactly midnight, only to retire at once to bed.
He never used the cosy chambers which the Reform provides for its favoured members. He passed ten hours out of the twenty-four in Saville Row, either in sleeping or making his toilet. When he chose to take a walk it was with a regular step in the entrance hall with its mosaic flooring, or in the circular gallery with its dome supported by twenty red porphyry Ionic columns, and illumined by blue painted windows.
When he breakfasted or dined all the resources of the club--its kitchens and pantries, its buttery and dairy--aided to crowd his table with their most succulent stores; he was served by the gravest waiters, in dress coats, and shoes with swan-skin soles, who proffered the viands in special porcelain, and on the finest linen; club decanters, of a lost mould, contained his sherry, his port, and his cinnamon-spiced claret; while his beverages were refreshingly cooled with ice, brought at great cost from the American lakes."""

# Tokenize the data
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])
total_words = len(tokenizer.word_index) + 1

# Create input sequences using tokens
input_sequences = []
for line in data.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# Create predictors and label
xs, labels = input_sequences[:,:-1],input_sequences[:,-1]
ys = tf.keras.utils.to_categorical(labels, num_classes=total_words)

# Define the Transformer model
embedding_dim = 128
dff = 128
vocab_size = total_words
dropout_rate = 0.1
num_layers = 2
num_heads = 2  # Define the number of attention heads

inputs = Input(shape=(max_sequence_len-1,))
embedding_layer = Embedding(vocab_size, embedding_dim)(inputs)

# Self-attention layer
attention_output = MultiHeadAttention(num_heads=num_heads, key_dim=embedding_dim//num_heads)(embedding_layer, embedding_layer)
attention_output = Dropout(dropout_rate)(attention_output)
attention_output = LayerNormalization(epsilon=1e-6)(attention_output + embedding_layer)

# Feed-forward network
ffn_output = Dense(dff, activation='relu')(attention_output)
ffn_output = Dense(embedding_dim)(ffn_output)
ffn_output = Dropout(dropout_rate)(ffn_output)
transformer_output = LayerNormalization(epsilon=1e-6)(ffn_output + attention_output)

outputs = GlobalMaxPooling1D()(transformer_output)
outputs = Dense(vocab_size, activation='softmax')(outputs)

model = Model(inputs=inputs, outputs=outputs)

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])

# Train the model
history = model.fit(xs, ys, epochs=100, verbose=1)

# Save the model
model.save('text_prediction_model.h5')


Epoch 1/100
27/27 [==============================] - 4s 82ms/step - loss: 6.0934 - accuracy: 0.0488
Epoch 2/100
27/27 [==============================] - 2s 77ms/step - loss: 5.4364 - accuracy: 0.0678
Epoch 3/100
27/27 [==============================] - 3s 126ms/step - loss: 5.3606 - accuracy: 0.0678
Epoch 4/100
27/27 [==============================] - 2s 75ms/step - loss: 5.3154 - accuracy: 0.0678
Epoch 5/100
27/27 [==============================] - 2s 77ms/step - loss: 5.2900 - accuracy: 0.0666
Epoch 6/100
27/27 [==============================] - 2s 78ms/step - loss: 5.2043 - accuracy: 0.0785
Epoch 7/100
27/27 [==============================] - 2s 77ms/step - loss: 5.0400 - accuracy: 0.0773
Epoch 8/100
27/27 [==============================] - 2s 89ms/step - loss: 4.8436 - accuracy: 0.0820
Epoch 9/100
27/27 [==============================] - 3s 119ms/step - loss: 4.5846 - accuracy: 0.0987
Epoch 10/100
27/27 [==============================] - 2s 79ms/step - loss: 4.2744 - accuracy: 0.09

In [21]:
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Replace "your_text_data_here" with your actual text data
test_data = """Mr. Phileas Fogg lived, in 1872, at No. 7, Saville Row, Burlington Gardens, the house in which Sheridan died in 1814.
He was one of the most noticeable members of the Reform Club, though he seemed always to avoid attracting attention; an enigmatical personage, about whom little was known, except that he was a polished man of the world.
People said that he resembled Byron--at least that his head was Byronic; but he was a bearded, tranquil Byron, who might live on a thousand years without growing old.
Certainly an Englishman, it was more doubtful whether Phileas Fogg was a Londoner.
He was never seen on 'Change, nor at the Bank, nor in the counting-rooms of the "City"; no ships ever came into London docks of which he was the owner; he had no public employment; he had never been entered at any of the Inns of Court, either at the Temple, or Lincoln's Inn, or Gray's Inn; nor had his voice ever resounded in the Court of Chancery, or in the Exchequer, or the Queen's Bench, or the Ecclesiastical Courts.
He certainly was not a manufacturer; nor was he a merchant or a gentleman farmer.
His name was strange to the scientific and learned societies, and he never was known to take part in the sage deliberations of the Royal Institution or the London Institution, the Artisan's Association, or the Institution of Arts and Sciences.
He belonged, in fact, to none of the numerous societies which swarm in the English capital, from the Harmonic to that of the Entomologists, founded mainly for the purpose of abolishing pernicious insects.
Was Phileas Fogg rich? Undoubtedly. But those who knew him best could not imagine how he had made his fortune, and Mr. Fogg was the last person to whom to apply for the information.He was not lavish, nor, on the contrary, avaricious; for, whenever he knew that money was needed for a noble, useful, or benevolent purpose, he supplied it quietly and sometimes anonymously. He was, in short, the least communicative of men.
He talked very little, and seemed all the more mysterious for his taciturn manner. His daily habits were quite open to observation; but whatever he did was so exactly the same thing that he had always done before, that the wits of the curious were fairly puzzled.
Had he travelled? It was likely, for no one seemed to know the world more familiarly; there was no spot so secluded that he did not appear to have an intimate acquaintance with it. He often corrected, with a few clear words, the thousand conjectures advanced by members of the club as to lost and unheard-of travellers, pointing out the true probabilities, and seeming as if gifted with a sort of second sight, so often did events justify his predictions. He must have travelled everywhere, at least in the spirit.
It was at least certain that Phileas Fogg had not absented himself from London for many years. Those who were honoured by a better acquaintance with him than the rest, declared that nobody could pretend to have ever seen him anywhere else. His sole pastimes were reading the papers and playing whist.
He often won at this game, which, as a silent one, harmonised with his nature; but his winnings never went into his purse, being reserved as a fund for his charities.
Mr. Fogg played, not to win, but for the sake of playing. The game was in his eyes a contest, a struggle with a difficulty, yet a motionless, unwearying struggle, congenial to his tastes.
Phileas Fogg was not known to have either wife or children, which may happen to the most honest people; either relatives or near friends, which is certainly more unusual. He lived alone in his house in Saville Row, whither none penetrated. A single domestic sufficed to serve him. He breakfasted and dined at the club, at hours mathematically fixed, in the same room, at the same table, never taking his meals with other members, much less bringing a guest with him; and went home at exactly midnight, only to retire at once to bed.
He never used the cosy chambers which the Reform provides for its favoured members. He passed ten hours out of the twenty-four in Saville Row, either in sleeping or making his toilet. When he chose to take a walk it was with a regular step in the entrance hall with its mosaic flooring, or in the circular gallery with its dome supported by twenty red porphyry Ionic columns, and illumined by blue painted windows.
When he breakfasted or dined all the resources of the club--its kitchens and pantries, its buttery and dairy--aided to crowd his table with their most succulent stores; he was served by the gravest waiters, in dress coats, and shoes with swan-skin soles, who proffered the viands in special porcelain, and on the finest linen; club decanters, of a lost mould, contained his sherry, his port, and his cinnamon-spiced claret; while his beverages were refreshingly cooled with ice, brought at great cost from the American lakes."""

# Load the tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])  # Make sure to use the same data used for training
total_words = len(tokenizer.word_index) + 1

# Function to generate text prediction
def generate_text(seed_text, next_words, max_sequence_len):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        predicted = model.predict_classes(token_list, verbose=0)
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

# Test the model
seed_text = "Phileas Fogg was not known to have"
next_words = 25
max_sequence_len = max([len(x) for x in input_sequences])  # Get the maximum sequence length

for _ in range(next_words):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted_probs = model.predict(token_list, verbose=0)
    predicted_index = np.argmax(predicted_probs)
    output_word = ""
    for word, index in tokenizer.word_index.items():
        if index == predicted_index:
            output_word = word
            break
    seed_text += " " + output_word

print(seed_text)



Phileas Fogg was not known to have either wife or children which may happen to to to most honest people or or either either or relatives or or or or or or


In [ ]:
# below code with test validation

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Dense, GlobalMaxPooling1D, Embedding, Dropout, LayerNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import MultiHeadAttention, Dense, Dropout, LayerNormalization
import numpy as np

# Replace the placeholder with your data
data = "your_text_data_here"

# Tokenize the data
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])
total_words = len(tokenizer.word_index) + 1

# Create input sequences using tokens
input_sequences = []
for line in data.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# Create predictors and label
xs, labels = input_sequences[:,:-1], input_sequences[:,-1]
ys = tf.keras.utils.to_categorical(labels, num_classes=total_words)

# Split data into training and validation sets
from sklearn.model_selection import train_test_split
x_train, x_val, y_train, y_val = train_test_split(xs, ys, test_size=0.2, random_state=42)

# Define the Transformer model
embedding_dim = 128
dff = 128
vocab_size = total_words
dropout_rate = 0.1
num_layers = 2
num_heads = 2  # Define the number of attention heads

inputs = Input(shape=(max_sequence_len-1,))
embedding_layer = Embedding(vocab_size, embedding_dim)(inputs)

# Self-attention layer
attention_output = MultiHeadAttention(num_heads=num_heads, key_dim=embedding_dim//num_heads)(embedding_layer, embedding_layer)
attention_output = Dropout(dropout_rate)(attention_output)
attention_output = LayerNormalization(epsilon=1e-6)(attention_output + embedding_layer)

# Feed-forward network
ffn_output = Dense(dff, activation='relu')(attention_output)
ffn_output = Dense(embedding_dim)(ffn_output)
ffn_output = Dropout(dropout_rate)(ffn_output)
transformer_output = LayerNormalization(epsilon=1e-6)(ffn_output + attention_output)

outputs = GlobalMaxPooling1D()(transformer_output)
outputs = Dense(vocab_size, activation='softmax')(outputs)

model = Model(inputs=inputs, outputs=outputs)

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])

# Train the model
history = model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=100, verbose=1)

# Save the model
model.save('text_prediction_model.h5')
